# 32 — Energy-Sweep Pipeline (`pipelines.energy_sweep`)

XANES-like beam-energy sweeps need integration of *every* image at the
*current* energy, with the calibration tracking the wavelength change.

`pipelines.energy_sweep.run_energy_sweep` is the batch wrapper that
takes (image, wavelength) pairs and produces a stack of integrated
profiles, properly handling the per-frame Q remapping so all profiles
share a common Q grid.

In [ ]:
import os; os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import math, numpy as np, torch, matplotlib.pyplot as plt

In [ ]:
from midas_integrate_v2.pipelines import energy_sweep

# Inputs: a list of (image, wavelength_A) pairs + a calibration spec
# Output: stacked 1-D profiles on a common Q grid + per-frame metadata
print('run_energy_sweep(spec, image_list, wavelengths, Q_grid_invA=...) -> EnergySweepResult')
print('  .profiles  : [n_frames, n_Q_bins]')
print('  .Q_grid    : [n_Q_bins]')
print('  .metadata  : per-frame energy, integration time, etc.')

## Use cases

* **XANES-EXAFS** edge scans with simultaneous diffraction
* **Anomalous scattering** Q-resolved structure-factor extraction
* **Resonant scattering** experiments (REXS / RXES)

The pipeline assumes the calibration geometry (BC, tilts, distortion)
stays fixed — only `Wavelength` changes per frame. If geometry drifts
(thermal), combine with NB 19 (`fit_drift_trajectory`) first.